**Выполнено: вариант 7.** Предметная область — сортировочная станция; фабрика создаёт тип отцепа; стратегия варьирует очерёдность расформирования; наблюдатели получают событие «состав расформирован».

# Песочница паттернов GoF: Strategy, Factory Method, Observer

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java. Каждый пример представлен **одним файлом** `.java` и запускается без отдельной компиляции командой `java Файл.java` (запуск исходного файла, Java 11 и выше).

**Идея задания.** Паттерн изучается не по описанию, а по разнице: сначала запускается код «до» — работающий, но жёстко связанный; затем тот же сценарий переписывается на паттерн, и проверки доказывают, что **поведение не изменилось**, а **новое требование добавляется без правки старого кода**.

**Состав работы**

| Раздел | Паттерн | Что делает студент |
|---|---|---|
| 2 | Strategy | **образец** — разобран полностью, нужно только запустить и разобраться |
| 3 | Factory Method | задача: самостоятельно переписать код «до» |
| 4 | Observer | задача: самостоятельно переписать код «до» |
| 5–6 | — | сравнение, выводы, контрольные вопросы |

**Правила оформления, от которых зависит автопроверка**

1. Имена файлов строго: `TariffAfter.java`, `FactoryAfter.java`, `ObserverAfter.java`.
2. В каждом файле **первым** идёт открытый класс с методом `main` — иначе запуск исходного файла не работает.
3. Каждый файл сам себя проверяет: печатает строки `[OK]` / `[FAIL]` и бросает `AssertionError`, если хотя бы одна проверка не пройдена.
4. Идентификаторы — латиницей и описательно (`tariffStrategy`, а не `ts`), комментарии — по-русски.
5. К каждому паттерну — диаграмма классов в файле `*.puml`.

**Сдача.** Все файлы `.java`, `.puml`, сам блокнот и вывод ячейки самопроверки — в pull request.

## 1. Подготовка среды

In [1]:
# 1.1 Проверить, какая Java доступна в среде.
# Нужна версия 11 или выше: только с неё работает запуск одного исходного файла (java Файл.java).
!java -version
!javac -version

openjdk version "17.0.21" 2026-10-20
OpenJDK Runtime Environment Temurin-17.0.21+9 (build 17.0.21+9)
OpenJDK 64-Bit Server VM Temurin-17.0.21+9 (build 17.0.21+9, mixed mode, sharing)


Если версия ниже 11, запуск одного исходного файла недоступен — всюду в блокноте вместо `!java Файл.java` используйте два шага: `!javac Файл.java && java Файл`.

Проверка «живого» запуска (ячейка 1.2 записывает файл, 1.3 — запускает):

In [2]:
%%writefile Hello.java
public class Hello {
    public static void main(String[] args) {
        System.out.println("Среда готова, Java работает: " + System.getProperty("java.version"));
    }
}

In [3]:
!java Hello.java

╨б╤А╨╡╨┤╨░ ╨│╨╛╤В╨╛╨▓╨░, Java ╤А╨░╨▒╨╛╤В╨░╨╡╤В: 17.0.21


---

## 2. Образец: Strategy (разобран полностью)

**Сценарий.** Расчёт платы за перевозку груза. Плата зависит от вида отправки: повагонная, контейнерная, маршрутная. Формулы учебные.

### 2.1 Код «до»: один метод знает про все виды отправок

In [4]:
%%writefile TariffBefore.java
// Код «до»: вид отправки разбирается условным оператором внутри одного метода.
public class TariffBefore {

    public static void main(String[] args) {
        System.out.printf("повагонная   : %10.2f%n", cost("WAGON", 60, 500));
        System.out.printf("контейнерная : %10.2f%n", cost("CONTAINER", 24, 500));
        System.out.printf("маршрутная   : %10.2f%n", cost("BLOCK_TRAIN", 3000, 500));
        System.out.printf("рефрижератор : %10.2f%n", cost("REFRIGERATED", 60, 500));
    }

    static double cost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;   // про новый вид отправки метод не знает
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

In [5]:
!java TariffBefore.java

повагонная   :     510,00

контейнерная :     508,00

маршрутная   :   10500,00

рефрижератор :       0,00



### 2.2 Разбор: что именно плохо

Запустив код «до», обратите внимание:

* рефрижераторная отправка дала **0.00** — метод о ней не знает, и единственный способ это исправить — залезть внутрь уже работающего метода;
* каждый новый вид отправки удлиняет один и тот же метод, то есть нарушает **принцип открытости/закрытости** (открыт для расширения, закрыт для изменения);
* метод одновременно отвечает за выбор варианта и за сам расчёт — нарушение **единственной ответственности**;
* протестировать один вид отправки отдельно от остальных нельзя.

**Что делает Strategy.** Каждый вариант расчёта становится отдельным классом с общим интерфейсом, а вызывающий код работает с интерфейсом и больше не знает перечня вариантов.

### 2.3 Код «после»: вариант расчёта вынесен в стратегию

In [6]:
%%writefile TariffAfter.java
// Код «после»: каждый вид отправки — отдельная стратегия расчёта.
import java.util.LinkedHashMap;
import java.util.Map;

public class TariffAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {

        // Каталог стратегий: вызывающий код собирает его, но не содержит формул
        Map<String, TariffStrategy> tariffCatalog = new LinkedHashMap<>();
        tariffCatalog.put("WAGON", new WagonTariff());
        tariffCatalog.put("CONTAINER", new ContainerTariff());
        tariffCatalog.put("BLOCK_TRAIN", new BlockTrainTariff());

        TariffCalculator tariffCalculator = new TariffCalculator();

        System.out.println("--- Расчёт по каталогу стратегий (масса 60 т, расстояние 500 км) ---");
        for (Map.Entry<String, TariffStrategy> catalogEntry : tariffCatalog.entrySet()) {
            double calculatedCost = tariffCalculator.calculate(catalogEntry.getValue(), 60, 500);
            System.out.printf("%-12s %-28s %10.2f%n",
                    catalogEntry.getKey(), catalogEntry.getValue().title(), calculatedCost);
        }

        System.out.println();
        System.out.println("--- Характеризующие тесты: поведение совпадает с кодом «до» ---");
        check("повагонная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("WAGON"), 60, 500),
                           legacyCost("WAGON", 60, 500)));
        check("контейнерная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("CONTAINER"), 24, 500),
                           legacyCost("CONTAINER", 24, 500)));
        check("маршрутная отправка",
                equalMoney(tariffCalculator.calculate(tariffCatalog.get("BLOCK_TRAIN"), 3000, 500),
                           legacyCost("BLOCK_TRAIN", 3000, 500)));

        System.out.println();
        System.out.println("--- Тест открытости/закрытости: новый вид отправки без правки существующих классов ---");
        TariffStrategy refrigeratedTariff = new TariffStrategy() {
            public double cost(double massTons, int distanceKm) {
                return 1.80 * massTons * distanceKm / 100.0 + 600.0;
            }
            public String title() {
                return "рефрижераторная отправка";
            }
        };
        double refrigeratedCost = tariffCalculator.calculate(refrigeratedTariff, 60, 500);
        System.out.printf("%-12s %-28s %10.2f%n", "REFRIGERATED", refrigeratedTariff.title(), refrigeratedCost);
        check("новая стратегия рассчитана, класс TariffCalculator не изменён", refrigeratedCost > 0.0);
        check("в коде «до» тот же вид отправки давал нуль", legacyCost("REFRIGERATED", 60, 500) == 0.0);

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    // --- служебное ---

    static boolean equalMoney(double firstValue, double secondValue) {
        return Math.abs(firstValue - secondValue) < 0.01;
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    // Старая реализация сохранена только как эталон поведения для характеризующих тестов
    static double legacyCost(String shipmentType, double massTons, int distanceKm) {
        double base;
        if (shipmentType.equals("WAGON")) {
            base = 1.20 * massTons * distanceKm / 100.0 + 150.0;
        } else if (shipmentType.equals("CONTAINER")) {
            base = 0.90 * massTons * distanceKm / 100.0 + 400.0;
        } else if (shipmentType.equals("BLOCK_TRAIN")) {
            base = 0.70 * massTons * distanceKm / 100.0;
        } else {
            base = 0.0;
        }
        return Math.round(base * 100.0) / 100.0;
    }
}

// --- участники паттерна Strategy ---

/** Стратегия: общий интерфейс всех вариантов расчёта. */
interface TariffStrategy {
    double cost(double massTons, int distanceKm);
    String title();
}

/** Контекст: умеет считать, но не знает, какие бывают виды отправок. */
class TariffCalculator {
    double calculate(TariffStrategy tariffStrategy, double massTons, int distanceKm) {
        double calculatedValue = tariffStrategy.cost(massTons, distanceKm);
        return Math.round(calculatedValue * 100.0) / 100.0;
    }
}

class WagonTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 1.20 * massTons * distanceKm / 100.0 + 150.0;
    }
    public String title() {
        return "повагонная отправка";
    }
}

class ContainerTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.90 * massTons * distanceKm / 100.0 + 400.0;
    }
    public String title() {
        return "контейнерная отправка";
    }
}

class BlockTrainTariff implements TariffStrategy {
    public double cost(double massTons, int distanceKm) {
        return 0.70 * massTons * distanceKm / 100.0;
    }
    public String title() {
        return "маршрутная отправка";
    }
}

In [7]:
!java TariffAfter.java

--- Расчёт по каталогу стратегий (масса 60 т, расстояние 500 км) ---

WAGON        повагонная отправка              510,00

CONTAINER    контейнерная отправка            670,00

BLOCK_TRAIN  маршрутная отправка              210,00



--- Характеризующие тесты: поведение совпадает с кодом «до» ---

[OK]   повагонная отправка

[OK]   контейнерная отправка

[OK]   маршрутная отправка



--- Тест открытости/закрытости: новый вид отправки без правки существующих классов ---

REFRIGERATED рефрижераторная отправка        1140,00

[OK]   новая стратегия рассчитана, класс TariffCalculator не изменён

[OK]   в коде «до» тот же вид отправки давал нуль



Все проверки пройдены.



### 2.4 Диаграмма классов образца

Установка PlantUML занимает около минуты и выполняется один раз за сеанс.

In [8]:
!apt-get -qq install -y plantuml > /dev/null && echo "PlantUML установлен"

In [9]:
%%writefile strategy.puml
@startuml
title Strategy: расчёт платы за перевозку

interface TariffStrategy {
  +cost(massTons, distanceKm) : double
  +title() : String
}

class TariffCalculator {
  +calculate(tariffStrategy, massTons, distanceKm) : double
}

class WagonTariff
class ContainerTariff
class BlockTrainTariff

TariffCalculator o--> TariffStrategy : использует
TariffStrategy <|.. WagonTariff
TariffStrategy <|.. ContainerTariff
TariffStrategy <|.. BlockTrainTariff
@enduml

In [10]:
!plantuml -tpng strategy.puml
from IPython.display import Image
Image("strategy.png")

### 2.5 Вопросы к образцу (ответы)

1. **Какие классы пришлось бы изменить при добавлении рефрижераторной отправки?**
   В коде «до» пришлось бы править метод `cost` класса `TariffBefore`: добавить ещё одну ветку `else if`. Это изменение уже работающего кода. В коде «после» существующие классы не меняются вообще: добавляется новый класс `RefrigeratedTariff` (или анонимная реализация `TariffStrategy`) и одна строка регистрации в каталоге внутри `main`. Расчётный класс `TariffCalculator` остаётся закрытым для изменений.

2. **Зачем в файле оставлен метод `legacyCost`?**
   Это эталон поведения старого кода для характеризующих тестов. Сравнение результата новой стратегии с результатом старого метода доказывает, что рефакторинг не изменил уже работающее поведение для трёх исходных видов отправок. После проверки расчёт в "`после`" этим методом не пользуется — он нужен только как тестовый ориентир.

3. **Что будет, если в каталоге нет запрошенного вида отправки и как это обработать?**
   `tariffCatalog.get(...)` вернёт `null`, и вызов `tariffStrategy.cost(...)` приведёт к `NullPointerException`. Такую ситуацию нужно обрабатывать явно: проверять наличие ключа и выбрасывать `IllegalArgumentException` с понятным сообщением и списком допустимых ключей, либо подставлять стратегию «неизвестный вид отправки», которая возвращает нулевую плату и пишет в журнал. Главное — не допускать «тихого» нуля, как в коде «до».

### 2.6 Вариант 7: Strategy на предметной области (дополнительно к образцу)

Раздел 2 выше — обязательный образец задания (расчёт платы). В варианте 7 стратегия варьирует **очерёдность расформирования** состава на сортировочной станции. Ниже тот же переход «до → после» показан на этой предметной области: три правила сортировки вагонов и четвёртое правило, добавленное без правки плана расформирования. Файлы `DismantlingBefore.java` / `DismantlingAfter.java` не заменяют обязательные `TariffBefore.java` / `TariffAfter.java`, а дополняют работу.

#### 2.6.1 Код «до»: правило выбирается цепочкой `if`

In [11]:
%%writefile DismantlingBefore.java
// Код «до» (вариант 7, сортировочная станция):
// очерёдность расформирования выбирается цепочкой условных операторов.
import java.util.ArrayList;
import java.util.Comparator;
import java.util.List;

public class DismantlingBefore {

    public static void main(String[] args) {
        List<TrainCar> cars = sampleCars();
        printOrder("по назначению", legacyOrder(cars, "BY_DESTINATION"));
        printOrder("по массе", legacyOrder(cars, "BY_WEIGHT"));
        printOrder("по типу вагона", legacyOrder(cars, "BY_TYPE"));
        printOrder("новое правило (приоритет)", legacyOrder(cars, "BY_PRIORITY"));
    }

    static void printOrder(String ruleName, List<String> carNumbers) {
        System.out.println(ruleName + ": " + String.join(" → ", carNumbers));
    }

    static List<String> legacyOrder(List<TrainCar> cars, String rule) {
        List<TrainCar> orderedCars = new ArrayList<>(cars);
        if (rule.equals("BY_DESTINATION")) {
            orderedCars.sort(Comparator.comparing((TrainCar car) -> car.destination)
                    .thenComparing((TrainCar car) -> car.number));
        } else if (rule.equals("BY_WEIGHT")) {
            orderedCars.sort(Comparator.comparing((TrainCar car) -> car.weightTons).reversed()
                    .thenComparing((TrainCar car) -> car.number));
        } else if (rule.equals("BY_TYPE")) {
            orderedCars.sort(Comparator.comparing((TrainCar car) -> car.carType)
                    .thenComparing((TrainCar car) -> car.number));
        }
        // Про новое правило метод не знает: порядок остаётся исходным.
        List<String> carNumbers = new ArrayList<>();
        for (TrainCar car : orderedCars) {
            carNumbers.add(car.number);
        }
        return carNumbers;
    }

    static List<TrainCar> sampleCars() {
        List<TrainCar> cars = new ArrayList<>();
        cars.add(new TrainCar("1", "Москва", 70, "крытый", false));
        cars.add(new TrainCar("2", "Брест", 55, "цистерна", true));
        cars.add(new TrainCar("3", "Москва", 62, "платформа", false));
        cars.add(new TrainCar("4", "Киев", 48, "крытый", false));
        cars.add(new TrainCar("5", "Брест", 75, "цистерна", false));
        return cars;
    }
}

/** Вагон состава: номер, направление, масса, тип и признак приоритетности. */
class TrainCar {
    final String number;
    final String destination;
    final int weightTons;
    final String carType;
    final boolean priority;

    TrainCar(String number, String destination, int weightTons, String carType, boolean priority) {
        this.number = number;
        this.destination = destination;
        this.weightTons = weightTons;
        this.carType = carType;
        this.priority = priority;
    }
}


In [12]:
!java DismantlingBefore.java

по назначению: 2 → 5 → 4 → 1 → 3

по массе: 5 → 1 → 3 → 2 → 4

по типу вагона: 1 → 4 → 3 → 2 → 5

новое правило (приоритет): 1 → 2 → 3 → 4 → 5



#### 2.6.2 Код «после»: каждая очерёдность — отдельная стратегия

In [13]:
%%writefile DismantlingAfter.java
// Вариант 7, сортировочная станция. Код «после»:
// Strategy — очерёдность расформирования состава.
import java.util.ArrayList;
import java.util.Comparator;
import java.util.LinkedHashMap;
import java.util.List;
import java.util.Map;

public class DismantlingAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {
        List<TrainCar> cars = sampleCars();
        List<String> originalOrder = carNumbers(cars);

        // Каталог стратегий: план расформирования не знает, какие правила существуют.
        Map<String, DismantlingOrderStrategy> strategyCatalog = new LinkedHashMap<>();
        strategyCatalog.put("BY_DESTINATION", new ByDestinationOrder());
        strategyCatalog.put("BY_WEIGHT", new ByWeightOrder());
        strategyCatalog.put("BY_TYPE", new ByCarTypeOrder());

        DismantlingPlan dismantlingPlan = new DismantlingPlan();

        System.out.println("--- Очерёдность расформирования по стратегиям ---");
        for (Map.Entry<String, DismantlingOrderStrategy> catalogEntry : strategyCatalog.entrySet()) {
            List<String> order = dismantlingPlan.makeOrder(cars, catalogEntry.getValue());
            System.out.printf("%-22s %-30s %s%n",
                    catalogEntry.getKey(), catalogEntry.getValue().title(), String.join(" → ", order));
        }

        System.out.println();
        System.out.println("--- Характеризующие тесты: поведение совпадает с кодом «до» ---");
        check("порядок по назначению совпадает с эталоном «до»",
                dismantlingPlan.makeOrder(cars, strategyCatalog.get("BY_DESTINATION"))
                        .equals(legacyOrder(cars, "BY_DESTINATION")));
        check("порядок по массе совпадает с эталоном «до»",
                dismantlingPlan.makeOrder(cars, strategyCatalog.get("BY_WEIGHT"))
                        .equals(legacyOrder(cars, "BY_WEIGHT")));
        check("порядок по типу вагона совпадает с эталоном «до»",
                dismantlingPlan.makeOrder(cars, strategyCatalog.get("BY_TYPE"))
                        .equals(legacyOrder(cars, "BY_TYPE")));
        check("в коде «до» новое правило оставляло исходный порядок",
                legacyOrder(cars, "BY_PRIORITY").equals(originalOrder));
        check("разные стратегии дают разную очерёдность",
                !dismantlingPlan.makeOrder(cars, strategyCatalog.get("BY_DESTINATION"))
                        .equals(dismantlingPlan.makeOrder(cars, strategyCatalog.get("BY_WEIGHT"))));

        System.out.println();
        System.out.println("--- Тест открытости/закрытости: новая очерёдность без правки плана ---");
        DismantlingOrderStrategy priorityOrder = new DismantlingOrderStrategy() {
            public List<String> order(List<TrainCar> carsToOrder) {
                return sortedNumbers(carsToOrder, Comparator.comparing((TrainCar car) -> car.priority)
                        .reversed().thenComparing((TrainCar car) -> car.number));
            }
            public String title() {
                return "сначала приоритетные вагоны";
            }
        };
        List<String> priorityOrderResult = dismantlingPlan.makeOrder(cars, priorityOrder);
        System.out.printf("%-22s %-30s %s%n", "BY_PRIORITY", priorityOrder.title(),
                String.join(" → ", priorityOrderResult));
        check("новая стратегия изменила порядок, DismantlingPlan не изменялся",
                !priorityOrderResult.equals(originalOrder));
        check("новая стратегия не потеряла ни одного вагона",
                priorityOrderResult.size() == originalOrder.size()
                        && priorityOrderResult.containsAll(originalOrder));

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    static List<String> carNumbers(List<TrainCar> cars) {
        List<String> numbers = new ArrayList<>();
        for (TrainCar car : cars) {
            numbers.add(car.number);
        }
        return numbers;
    }

    static List<String> sortedNumbers(List<TrainCar> cars, Comparator<TrainCar> comparator) {
        List<TrainCar> orderedCars = new ArrayList<>(cars);
        orderedCars.sort(comparator);
        return carNumbers(orderedCars);
    }

    // Эталон старого поведения для характеризующих тестов.
    static List<String> legacyOrder(List<TrainCar> cars, String rule) {
        List<TrainCar> orderedCars = new ArrayList<>(cars);
        if (rule.equals("BY_DESTINATION")) {
            orderedCars.sort(Comparator.comparing((TrainCar car) -> car.destination)
                    .thenComparing((TrainCar car) -> car.number));
        } else if (rule.equals("BY_WEIGHT")) {
            orderedCars.sort(Comparator.comparing((TrainCar car) -> car.weightTons).reversed()
                    .thenComparing((TrainCar car) -> car.number));
        } else if (rule.equals("BY_TYPE")) {
            orderedCars.sort(Comparator.comparing((TrainCar car) -> car.carType)
                    .thenComparing((TrainCar car) -> car.number));
        }
        return carNumbers(orderedCars);
    }

    static List<TrainCar> sampleCars() {
        List<TrainCar> cars = new ArrayList<>();
        cars.add(new TrainCar("1", "Москва", 70, "крытый", false));
        cars.add(new TrainCar("2", "Брест", 55, "цистерна", true));
        cars.add(new TrainCar("3", "Москва", 62, "платформа", false));
        cars.add(new TrainCar("4", "Киев", 48, "крытый", false));
        cars.add(new TrainCar("5", "Брест", 75, "цистерна", false));
        return cars;
    }
}

/** Стратегия: правило очерёдности расформирования. */
interface DismantlingOrderStrategy {
    List<String> order(List<TrainCar> cars);
    String title();
}

/** Контекст: строит порядок расформирования, не зная перечня правил. */
class DismantlingPlan {
    List<String> makeOrder(List<TrainCar> cars, DismantlingOrderStrategy strategy) {
        return strategy.order(cars);
    }
}

class ByDestinationOrder implements DismantlingOrderStrategy {
    public List<String> order(List<TrainCar> cars) {
        return DismantlingAfter.sortedNumbers(cars,
                Comparator.comparing((TrainCar car) -> car.destination)
                        .thenComparing((TrainCar car) -> car.number));
    }
    public String title() {
        return "сначала по назначению";
    }
}

class ByWeightOrder implements DismantlingOrderStrategy {
    public List<String> order(List<TrainCar> cars) {
        return DismantlingAfter.sortedNumbers(cars,
                Comparator.comparing((TrainCar car) -> car.weightTons).reversed()
                        .thenComparing((TrainCar car) -> car.number));
    }
    public String title() {
        return "сначала тяжёлые вагоны";
    }
}

class ByCarTypeOrder implements DismantlingOrderStrategy {
    public List<String> order(List<TrainCar> cars) {
        return DismantlingAfter.sortedNumbers(cars,
                Comparator.comparing((TrainCar car) -> car.carType)
                        .thenComparing((TrainCar car) -> car.number));
    }
    public String title() {
        return "сначала по типу вагона";
    }
}

/** Вагон состава: номер, направление, масса, тип и признак приоритетности. */
class TrainCar {
    final String number;
    final String destination;
    final int weightTons;
    final String carType;
    final boolean priority;

    TrainCar(String number, String destination, int weightTons, String carType, boolean priority) {
        this.number = number;
        this.destination = destination;
        this.weightTons = weightTons;
        this.carType = carType;
        this.priority = priority;
    }
}


In [14]:
!java DismantlingAfter.java

--- Очерёдность расформирования по стратегиям ---

BY_DESTINATION         сначала по назначению          2 → 5 → 4 → 1 → 3

BY_WEIGHT              сначала тяжёлые вагоны         5 → 1 → 3 → 2 → 4

BY_TYPE                сначала по типу вагона         1 → 4 → 3 → 2 → 5



--- Характеризующие тесты: поведение совпадает с кодом «до» ---

[OK]   порядок по назначению совпадает с эталоном «до»

[OK]   порядок по массе совпадает с эталоном «до»

[OK]   порядок по типу вагона совпадает с эталоном «до»

[OK]   в коде «до» новое правило оставляло исходный порядок

[OK]   разные стратегии дают разную очерёдность



--- Тест открытости/закрытости: новая очерёдность без правки плана ---

BY_PRIORITY            сначала приоритетные вагоны    2 → 1 → 3 → 4 → 5

[OK]   новая стратегия изменила порядок, DismantlingPlan не изменялся

[OK]   новая стратегия не потеряла ни одного вагона



Все проверки пройдены.



#### 2.6.3 Диаграмма классов варианта 7

In [15]:
%%writefile strategy_laba7.puml
@startuml
title Strategy: очерёдность расформирования состава (вариант 7)

interface DismantlingOrderStrategy {
  +order(cars) : List<String>
  +title() : String
}

class DismantlingPlan {
  +makeOrder(cars, strategy) : List<String>
}

class ByDestinationOrder
class ByWeightOrder
class ByCarTypeOrder

class TrainCar {
  -number : String
  -destination : String
  -weightTons : int
  -carType : String
  -priority : boolean
}

DismantlingPlan o--> DismantlingOrderStrategy : использует
DismantlingOrderStrategy <|.. ByDestinationOrder
DismantlingOrderStrategy <|.. ByWeightOrder
DismantlingOrderStrategy <|.. ByCarTypeOrder
DismantlingPlan ..> TrainCar : упорядочивает
@enduml

In [16]:
!plantuml -tpng strategy_laba7.puml
from IPython.display import Image
Image("strategy_laba7.png")

**Вывод по доп. задаче (Strategy, вариант 7).** В коде «до» правило порядка расформирования выбиралось цепочкой `if` в методе `legacyOrder`, а неизвестное правило молча оставляло исходный порядок. После рефакторинга `DismantlingPlan` работает с интерфейсом `DismantlingOrderStrategy` и не знает перечня правил: правила «по назначению», «по массе» и «по типу вагона» стали классами `ByDestinationOrder`, `ByWeightOrder`, `ByCarTypeOrder`. Четвёртое правило (сначала приоритетные вагоны) добавлено анонимной реализацией и проверено характеризующими тестами: старые три порядка совпадают с эталоном, новый порядок отличается и не теряет вагоны. Класс `DismantlingPlan` при этом не изменялся.

## 3. Задача 1: Factory Method

**Сценарий (вариант 7).** Сортировочная станция создаёт отцепы разных типов.

### 3.1 Код «до» (вариант 7 — сортировочная станция)

Тип отцепа разбирается строковым условием внутри одного метода. Четвёртый тип (местный отцеп) не обрабатывается — это проявление нарушения принципа открытости/закрытости.

In [17]:
%%writefile FactoryBefore.java
// Код «до» (вариант 7, сортировочная станция):
// тип отцепа разбирается условным оператором внутри одного метода.
public class FactoryBefore {

    public static void main(String[] args) {
        System.out.println(describe("BY_DESTINATION", 8, 68));
        System.out.println(describe("TRANSIT", 5, 71));
        System.out.println(describe("REPAIR", 3, 66));
        System.out.println(describe("LOCAL", 4, 70));   // новый тип — обработчика нет
    }

    static String describe(String otsepType, int carCount, double capacityPerCarTons) {
        double totalCapacityTons = carCount * capacityPerCarTons;
        if (otsepType.equals("BY_DESTINATION")) {
            return "Отцеп по назначению: " + carCount + " вагонов, грузоподъёмность "
                    + totalCapacityTons + " т, следует на сортировочный путь по направлению";
        } else if (otsepType.equals("TRANSIT")) {
            return "Транзитный отцеп: " + carCount + " вагонов, грузоподъёмность "
                    + totalCapacityTons + " т, следует без переработки";
        } else if (otsepType.equals("REPAIR")) {
            return "Ремонтный отцеп: " + carCount + " вагонов, грузоподъёмность "
                    + totalCapacityTons + " т, вагоны направлены в вагонное депо";
        }
        return "Неизвестный тип отцепа: " + otsepType;
    }
}


In [18]:
!java FactoryBefore.java

Отцеп по назначению: 8 вагонов, грузоподъёмность 544.0 т, следует на сортировочный путь по направлению

Транзитный отцеп: 5 вагонов, грузоподъёмность 355.0 т, следует без переработки

Ремонтный отцеп: 3 вагонов, грузоподъёмность 198.0 т, вагоны направлены в вагонное депо

Неизвестный тип отцепа: LOCAL



### 3.2 Задание (вариант 7)

Переписать сценарий на **Factory Method** в файле `FactoryAfter.java`. Требования задания адаптированы под вариант 7:

| Требование шаблона | Реализация варианта 7 |
|---|---|
| продукт `Wagon` с `description()` и `capacityTons()` | продукт `Otsep` (отцеп) с `description()`, `carCount()`, `capacityTons()` |
| `CoveredWagon`, `Platform`, `TankWagon` | `DestinationOtsep`, `TransitOtsep`, `RepairOtsep` |
| создатель `WagonCreator.createWagon(...)` | создатель `OtsepCreator.createOtsep(carCount, capacityPerCarTons)` |
| четвёртый тип `HopperWagon` | четвёртый тип `LocalOtsep` + `LocalOtsepCreator` |

1. Вызывающий код (`main`) работает **только** с типами `OtsepCreator` и `Otsep` и не содержит ни одного `if` по типу отцепа.
2. Четвёртый тип (`LocalOtsep`, местный отцеп) добавляется новыми классами, без правки ранее написанных классов; это видно по коду.
3. Проверки в файле (метод `check`), минимум четыре: описание не пустое и содержит число вагонов и грузоподъёмность; тип созданного объекта соответствует ожидаемому создателю; четвёртый тип работает наравне с остальными; старый код не знает новый тип (характеризующий тест).
4. При неуспешной проверке — `throw new AssertionError(...)`.

**Структура файла:** первым идёт `public class FactoryAfter` с `main`, после него — интерфейс продукта, абстрактный создатель, конкретные продукты и создатели (как в `TariffAfter.java`).

In [19]:
%%writefile FactoryAfter.java
// Вариант 7, сортировочная станция. Код «после»:
// Factory Method — фабрика создаёт тип отцепа.
import java.util.Arrays;
import java.util.List;

public class FactoryAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {
        // Вызывающий код работает только с типами OtsepCreator и Otsep.
        // Четвёртый создатель добавлен новым классом, старые классы не тронуты.
        List<OtsepCreator> otsepCreators = Arrays.asList(
                new DestinationOtsepCreator(),
                new TransitOtsepCreator(),
                new RepairOtsepCreator(),
                new LocalOtsepCreator());

        System.out.println("--- Отцепы, созданные фабричным методом ---");
        for (OtsepCreator otsepCreator : otsepCreators) {
            Otsep otsep = otsepCreator.createOtsep(4, 68.0);
            System.out.printf("%-24s %-32s %d вагонов, %.1f т%n",
                    otsepCreator.title(), otsep.description(), otsep.carCount(), otsep.capacityTons());
        }

        Otsep destinationOtsep = otsepCreators.get(0).createOtsep(4, 68.0);
        Otsep transitOtsep = otsepCreators.get(1).createOtsep(5, 71.0);
        Otsep repairOtsep = otsepCreators.get(2).createOtsep(2, 66.0);
        Otsep localOtsep = otsepCreators.get(3).createOtsep(3, 70.0);

        System.out.println();
        System.out.println("--- Проверки Factory Method (вариант 7) ---");
        check("описание отцепа по назначению не пустое и содержит число вагонов",
                !destinationOtsep.description().isEmpty()
                        && destinationOtsep.description().contains("4"));
        check("описание содержит общую грузоподъёмность отцепа",
                destinationOtsep.description().contains("272.0")
                        && destinationOtsep.capacityTons() == 272.0);
        check("созданный объект имеет тип DestinationOtsep",
                destinationOtsep instanceof DestinationOtsep);
        check("созданный объект имеет тип TransitOtsep",
                transitOtsep instanceof TransitOtsep);
        check("созданный объект имеет тип RepairOtsep",
                repairOtsep instanceof RepairOtsep);
        check("четвёртый тип (местный отцеп) работает наравне с остальными",
                localOtsep instanceof LocalOtsep
                        && localOtsep.description().contains("Местн")
                        && localOtsep.carCount() == 3);
        check("фабричный метод создаёт новый объект при каждом вызове",
                otsepCreators.get(0).createOtsep(4, 68.0)
                        != otsepCreators.get(0).createOtsep(4, 68.0));
        check("все создатели дают непустые описания",
                allDescriptionsAreNotEmpty(otsepCreators));
        check("код «до» не знал про местный отцеп (характеризующий тест)",
                legacyDescribe("LOCAL", 3, 70.0).startsWith("Неизвестный"));

        finishChecks();
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }

    // Проверку завершения выносим из main: в main нет никаких ветвлений по типу.
    static void finishChecks() {
        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
    }

    static boolean allDescriptionsAreNotEmpty(List<OtsepCreator> otsepCreators) {
        for (OtsepCreator otsepCreator : otsepCreators) {
            if (otsepCreator.createOtsep(1, 60.0).description().isEmpty()) {
                return false;
            }
        }
        return true;
    }

    // Старая реализация сохранена только как эталон поведения для характеризующего теста.
    static String legacyDescribe(String otsepType, int carCount, double capacityPerCarTons) {
        double totalCapacityTons = carCount * capacityPerCarTons;
        if (otsepType.equals("BY_DESTINATION")) {
            return "Отцеп по назначению: " + carCount + " вагонов, грузоподъёмность "
                    + totalCapacityTons + " т, следует на сортировочный путь по направлению";
        } else if (otsepType.equals("TRANSIT")) {
            return "Транзитный отцеп: " + carCount + " вагонов, грузоподъёмность "
                    + totalCapacityTons + " т, следует без переработки";
        } else if (otsepType.equals("REPAIR")) {
            return "Ремонтный отцеп: " + carCount + " вагонов, грузоподъёмность "
                    + totalCapacityTons + " т, вагоны направлены в вагонное депо";
        }
        return "Неизвестный тип отцепа: " + otsepType;
    }
}

/** Продукт: отцеп — группа вагонов, отделяемая от состава на сортировочной станции. */
interface Otsep {
    String description();
    int carCount();
    double capacityTons();
}

/** Создатель: фабричный метод возвращает отцеп нужного типа. */
abstract class OtsepCreator {
    abstract Otsep createOtsep(int carCount, double capacityPerCarTons);
    abstract String title();
}

class DestinationOtsep implements Otsep {
    private final int carCount;
    private final double capacityPerCarTons;

    DestinationOtsep(int carCount, double capacityPerCarTons) {
        this.carCount = carCount;
        this.capacityPerCarTons = capacityPerCarTons;
    }
    public String description() {
        return "Отцеп по назначению: " + carCount + " вагонов, грузоподъёмность "
                + capacityTons() + " т, следует на сортировочный путь по направлению";
    }
    public int carCount() {
        return carCount;
    }
    public double capacityTons() {
        return carCount * capacityPerCarTons;
    }
}

class TransitOtsep implements Otsep {
    private final int carCount;
    private final double capacityPerCarTons;

    TransitOtsep(int carCount, double capacityPerCarTons) {
        this.carCount = carCount;
        this.capacityPerCarTons = capacityPerCarTons;
    }
    public String description() {
        return "Транзитный отцеп: " + carCount + " вагонов, грузоподъёмность "
                + capacityTons() + " т, следует без переработки";
    }
    public int carCount() {
        return carCount;
    }
    public double capacityTons() {
        return carCount * capacityPerCarTons;
    }
}

class RepairOtsep implements Otsep {
    private final int carCount;
    private final double capacityPerCarTons;

    RepairOtsep(int carCount, double capacityPerCarTons) {
        this.carCount = carCount;
        this.capacityPerCarTons = capacityPerCarTons;
    }
    public String description() {
        return "Ремонтный отцеп: " + carCount + " вагонов, грузоподъёмность "
                + capacityTons() + " т, вагоны направлены в вагонное депо";
    }
    public int carCount() {
        return carCount;
    }
    public double capacityTons() {
        return carCount * capacityPerCarTons;
    }
}

class LocalOtsep implements Otsep {
    private final int carCount;
    private final double capacityPerCarTons;

    LocalOtsep(int carCount, double capacityPerCarTons) {
        this.carCount = carCount;
        this.capacityPerCarTons = capacityPerCarTons;
    }
    public String description() {
        return "Местный отцеп: " + carCount + " вагонов, грузоподъёмность "
                + capacityTons() + " т, подача на местный грузовой фронт";
    }
    public int carCount() {
        return carCount;
    }
    public double capacityTons() {
        return carCount * capacityPerCarTons;
    }
}

class DestinationOtsepCreator extends OtsepCreator {
    Otsep createOtsep(int carCount, double capacityPerCarTons) {
        return new DestinationOtsep(carCount, capacityPerCarTons);
    }
    String title() {
        return "создатель отцепа по назначению";
    }
}

class TransitOtsepCreator extends OtsepCreator {
    Otsep createOtsep(int carCount, double capacityPerCarTons) {
        return new TransitOtsep(carCount, capacityPerCarTons);
    }
    String title() {
        return "создатель транзитного отцепа";
    }
}

class RepairOtsepCreator extends OtsepCreator {
    Otsep createOtsep(int carCount, double capacityPerCarTons) {
        return new RepairOtsep(carCount, capacityPerCarTons);
    }
    String title() {
        return "создатель ремонтного отцепа";
    }
}

class LocalOtsepCreator extends OtsepCreator {
    Otsep createOtsep(int carCount, double capacityPerCarTons) {
        return new LocalOtsep(carCount, capacityPerCarTons);
    }
    String title() {
        return "создатель местного отцепа";
    }
}


In [20]:
!java FactoryAfter.java

--- Отцепы, созданные фабричным методом ---

создатель отцепа по назначению Отцеп по назначению: 4 вагонов, грузоподъёмность 272.0 т, следует на сортировочный путь по направлению 4 вагонов, 272,0 т

создатель транзитного отцепа Транзитный отцеп: 4 вагонов, грузоподъёмность 272.0 т, следует без переработки 4 вагонов, 272,0 т

создатель ремонтного отцепа Ремонтный отцеп: 4 вагонов, грузоподъёмность 272.0 т, вагоны направлены в вагонное депо 4 вагонов, 272,0 т

создатель местного отцепа Местный отцеп: 4 вагонов, грузоподъёмность 272.0 т, подача на местный грузовой фронт 4 вагонов, 272,0 т



--- Проверки Factory Method (вариант 7) ---

[OK]   описание отцепа по назначению не пустое и содержит число вагонов

[OK]   описание содержит общую грузоподъёмность отцепа

[OK]   созданный объект имеет тип DestinationOtsep

[OK]   созданный объект имеет тип TransitOtsep

[OK]   созданный объект имеет тип RepairOtsep

[OK]   четвёртый тип (местный отцеп) работает наравне с остальными

[OK]   фабричны

In [21]:
%%writefile factory.puml
@startuml
title Factory Method: создание отцепа на сортировочной станции (вариант 7)

interface Otsep {
  +description() : String
  +carCount() : int
  +capacityTons() : double
}

abstract class OtsepCreator {
  +{abstract}createOtsep(carCount, capacityPerCarTons) : Otsep
  +{abstract}title() : String
}

class DestinationOtsep
class TransitOtsep
class RepairOtsep
class LocalOtsep

class DestinationOtsepCreator
class TransitOtsepCreator
class RepairOtsepCreator
class LocalOtsepCreator

OtsepCreator <|-- DestinationOtsepCreator
OtsepCreator <|-- TransitOtsepCreator
OtsepCreator <|-- RepairOtsepCreator
OtsepCreator <|-- LocalOtsepCreator

Otsep <|.. DestinationOtsep
Otsep <|.. TransitOtsep
Otsep <|.. RepairOtsep
Otsep <|.. LocalOtsep

DestinationOtsepCreator ..> DestinationOtsep : создаёт
TransitOtsepCreator ..> TransitOtsep : создаёт
RepairOtsepCreator ..> RepairOtsep : создаёт
LocalOtsepCreator ..> LocalOtsep : создаёт
@enduml

In [22]:
!plantuml -tpng factory.puml
from IPython.display import Image
Image("factory.png")

**Вывод по задаче 1 (Factory Method, вариант 7).** В вызывающем коде вместо `new` и разбора строкового типа используется `OtsepCreator.createOtsep`: `main` работает только с интерфейсами `OtsepCreator` и `Otsep` и не содержит ни одного `if`/`switch`. Фабричный метод переносит решение «какой это отцеп» в специализированный создатель, поэтому местный отцеп добавлен новыми классами `LocalOtsep` и `LocalOtsepCreator` без правки прежних классов. Отличие от прямого `new`: клиент не зависит от конкретного класса продукта и может получить другую реализацию, заменив создателя, не меняя клиентский код. Фабрика излишня, если тип продукта единственный, не расширяется, а создание — простой вызов конструктора без дополнительной логики: тогда она лишь увеличивает число классов.

---

## 4. Задача 2: Observer

**Сценарий.** Станция фиксирует прибытие поезда и должна оповестить получателей: табло, журнал, рассылку. В коде «до» станция сама знает про всех получателей.

### 4.1 Код «до» (запустить, не изменяя)

In [23]:
%%writefile ObserverBefore.java
// Код «до» (вариант 7, сортировочная станция):
// источник события жёстко связан со всеми получателями.
public class ObserverBefore {

    public static void main(String[] args) {
        Station station = new Station("Гомель-Сортировочная");
        station.trainDismantled("3022", 62);
        station.trainDismantled("6701", 48);
    }
}

class Station {

    private final String stationName;

    Station(String stationName) {
        this.stationName = stationName;
    }

    void trainDismantled(String trainNumber, int carCount) {
        // Любой новый получатель требует правки этого метода
        System.out.println("[табло]  Состав " + trainNumber + " расформирован на станции "
                + stationName + " (" + carCount + " вагонов)");
        System.out.println("[журнал] " + stationName + ": зафиксировано расформирование состава "
                + trainNumber + ", вагонов " + carCount);
    }
}


In [24]:
!java ObserverBefore.java

[табло]  Состав 3022 расформирован на станции Гомель-Сортировочная (62 вагонов)

[журнал] Гомель-Сортировочная: зафиксировано расформирование состава 3022, вагонов 62

[табло]  Состав 6701 расформирован на станции Гомель-Сортировочная (48 вагонов)

[журнал] Гомель-Сортировочная: зафиксировано расформирование состава 6701, вагонов 48



### 4.2 Задание (вариант 7)

Переписать сценарий на **Observer** в файле `ObserverAfter.java`. Требования адаптированы под вариант 7:

| Требование шаблона | Реализация варианта 7 |
|---|---|
| интерфейс `ArrivalListener.onTrainArrived(...)` | интерфейс `DismantlingListener.onTrainDismantled(stationName, trainNumber, carCount)` |
| событие «поезд прибыл» | событие «состав расформирован» |
| субъект `Station` | субъект `Station` — сортировочная станция (имя сохранено) |
| табло, журнал, рассылка | `BoardDisplay`, `DismantlingJournal`, `NotificationService` |
| четвёртый наблюдатель | `AsuIntegration` — передача в АСУ |

1. Субъект `Station` имеет методы `addListener`, `removeListener`, `trainDismantled` и **не знает**, какие именно наблюдатели существуют.
2. Журнал накапливает записи, чтобы их можно было проверить.
3. Проверки в файле, минимум четыре: после одного события журнал получил ровно одну запись; после отписки наблюдателя его записи больше не появляются; все подписчики получили одно и то же событие; добавление четвёртого наблюдателя не потребовало изменения `Station`.
4. В выводе указан порядок получения события подписчиками.

In [25]:
%%writefile ObserverAfter.java
// Вариант 7, сортировочная станция. Код «после»:
// Observer — событие «состав расформирован» получают подписчики.
import java.util.ArrayList;
import java.util.Arrays;
import java.util.List;

public class ObserverAfter {

    private static int failedChecks = 0;

    public static void main(String[] args) {
        Station station = new Station("Гомель-Сортировочная");

        BoardDisplay boardDisplay = new BoardDisplay();
        DismantlingJournal dismantlingJournal = new DismantlingJournal();
        NotificationService notificationService = new NotificationService();
        AsuIntegration asuIntegration = new AsuIntegration();   // четвёртый наблюдатель

        station.addListener(boardDisplay);
        station.addListener(dismantlingJournal);
        station.addListener(notificationService);

        System.out.println("--- Расформирование состава 3022: подписаны три наблюдателя ---");
        station.trainDismantled("3022", 62);
        List<String> firstDeliveryOrder = station.deliveryOrder();
        printOrder("Порядок получения события (1-е событие)", firstDeliveryOrder);
        System.out.println("Записей в журнале: " + dismantlingJournal.recordCount());

        System.out.println();
        System.out.println("--- Проверки Observer (вариант 7) ---");
        check("после одного события журнал получил ровно одну запись",
                dismantlingJournal.recordCount() == 1);
        check("все подписчики получили одно и то же событие",
                boardDisplay.lastEvent().equals("Гомель-Сортировочная|3022|62")
                        && notificationService.lastEvent().equals("Гомель-Сортировочная|3022|62")
                        && dismantlingJournal.records().get(0).contains("3022"));

        String boardEventBeforeUnsubscribe = boardDisplay.lastEvent();

        // Отписка и подключение четвёртого наблюдателя: класс Station не меняется.
        station.removeListener(boardDisplay);
        station.addListener(asuIntegration);

        System.out.println();
        System.out.println("--- Расформирование состава 6701: табло отписано, подключена АСУ ---");
        station.trainDismantled("6701", 48);
        printOrder("Порядок получения события (2-е событие)", station.deliveryOrder());
        System.out.println("Записей в журнале: " + dismantlingJournal.recordCount());

        check("после отписки табло больше не получает события",
                boardDisplay.lastEvent().equals(boardEventBeforeUnsubscribe)
                        && boardDisplay.receiveCount() == 1);
        check("четвёртый наблюдатель (АСУ) получил событие без правки Station",
                asuIntegration.receiveCount() == 1
                        && asuIntegration.lastEvent().equals("Гомель-Сортировочная|6701|48"));
        check("журнал накопил обе записи, оставаясь подписанным",
                dismantlingJournal.recordCount() == 2
                        && dismantlingJournal.records().get(1).contains("6701"));
        check("порядок первого события: табло → журнал → рассылка",
                firstDeliveryOrder.equals(Arrays.asList("табло", "журнал", "рассылка")));
        check("порядок второго события: журнал → рассылка → АСУ",
                station.deliveryOrder().equals(Arrays.asList("журнал", "рассылка", "АСУ")));

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    static void printOrder(String caption, List<String> deliveryOrder) {
        System.out.println(caption + ": " + String.join(" → ", deliveryOrder));
    }

    static void check(String checkName, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + checkName);
        if (!passed) {
            failedChecks++;
        }
    }
}

/** Наблюдатель: получает событие «состав расформирован». */
interface DismantlingListener {
    void onTrainDismantled(String stationName, String trainNumber, int carCount);
    String listenerTitle();
}

/** Субъект: сортировочная станция. Не знает, какие именно наблюдатели существуют. */
class Station {

    private final String stationName;
    private final List<DismantlingListener> listeners = new ArrayList<>();
    private final List<String> currentDeliveryOrder = new ArrayList<>();

    Station(String stationName) {
        this.stationName = stationName;
    }

    void addListener(DismantlingListener listener) {
        listeners.add(listener);
    }

    void removeListener(DismantlingListener listener) {
        listeners.remove(listener);
    }

    void trainDismantled(String trainNumber, int carCount) {
        System.out.println("Событие: состав расформирован — станция " + stationName
                + ", поезд " + trainNumber + ", вагонов " + carCount);
        currentDeliveryOrder.clear();
        for (DismantlingListener listener : listeners) {
            listener.onTrainDismantled(stationName, trainNumber, carCount);
            currentDeliveryOrder.add(listener.listenerTitle());
        }
    }

    List<String> deliveryOrder() {
        return new ArrayList<>(currentDeliveryOrder);
    }
}

class BoardDisplay implements DismantlingListener {
    private String lastStationName = "";
    private String lastTrainNumber = "";
    private int lastCarCount = 0;
    private int receiveCount = 0;

    public void onTrainDismantled(String stationName, String trainNumber, int carCount) {
        lastStationName = stationName;
        lastTrainNumber = trainNumber;
        lastCarCount = carCount;
        receiveCount++;
        System.out.println("[табло]  Состав " + trainNumber + " расформирован на станции "
                + stationName + " (" + carCount + " вагонов)");
    }
    public String listenerTitle() {
        return "табло";
    }
    String lastEvent() {
        return lastStationName + "|" + lastTrainNumber + "|" + lastCarCount;
    }
    int receiveCount() {
        return receiveCount;
    }
}

class DismantlingJournal implements DismantlingListener {
    private final List<String> records = new ArrayList<>();

    public void onTrainDismantled(String stationName, String trainNumber, int carCount) {
        String record = stationName + ": состав " + trainNumber
                + " расформирован (" + carCount + " вагонов)";
        records.add(record);
        System.out.println("[журнал] " + record);
    }
    public String listenerTitle() {
        return "журнал";
    }
    List<String> records() {
        return new ArrayList<>(records);
    }
    int recordCount() {
        return records.size();
    }
}

class NotificationService implements DismantlingListener {
    private String lastStationName = "";
    private String lastTrainNumber = "";
    private int lastCarCount = 0;

    public void onTrainDismantled(String stationName, String trainNumber, int carCount) {
        lastStationName = stationName;
        lastTrainNumber = trainNumber;
        lastCarCount = carCount;
        System.out.println("[рассылка] Оповещение: состав " + trainNumber
                + " расформирован на станции " + stationName);
    }
    public String listenerTitle() {
        return "рассылка";
    }
    String lastEvent() {
        return lastStationName + "|" + lastTrainNumber + "|" + lastCarCount;
    }
}

class AsuIntegration implements DismantlingListener {
    private String lastStationName = "";
    private String lastTrainNumber = "";
    private int lastCarCount = 0;
    private int receiveCount = 0;

    public void onTrainDismantled(String stationName, String trainNumber, int carCount) {
        lastStationName = stationName;
        lastTrainNumber = trainNumber;
        lastCarCount = carCount;
        receiveCount++;
        System.out.println("[АСУ]     Передача в АСУ: станция " + stationName
                + ", состав " + trainNumber + " расформирован, вагонов " + carCount);
    }
    public String listenerTitle() {
        return "АСУ";
    }
    String lastEvent() {
        return lastStationName + "|" + lastTrainNumber + "|" + lastCarCount;
    }
    int receiveCount() {
        return receiveCount;
    }
}


In [26]:
!java ObserverAfter.java

--- Расформирование состава 3022: подписаны три наблюдателя ---

Событие: состав расформирован — станция Гомель-Сортировочная, поезд 3022, вагонов 62

[табло]  Состав 3022 расформирован на станции Гомель-Сортировочная (62 вагонов)

[журнал] Гомель-Сортировочная: состав 3022 расформирован (62 вагонов)

[рассылка] Оповещение: состав 3022 расформирован на станции Гомель-Сортировочная

Порядок получения события (1-е событие): табло → журнал → рассылка

Записей в журнале: 1



--- Проверки Observer (вариант 7) ---

[OK]   после одного события журнал получил ровно одну запись

[OK]   все подписчики получили одно и то же событие



--- Расформирование состава 6701: табло отписано, подключена АСУ ---

Событие: состав расформирован — станция Гомель-Сортировочная, поезд 6701, вагонов 48

[журнал] Гомель-Сортировочная: состав 6701 расформирован (48 вагонов)

[рассылка] Оповещение: состав 6701 расформирован на станции Гомель-Сортировочная

[АСУ]     Передача в АСУ: станция Гомель-Сортировочная, со

In [27]:
%%writefile observer.puml
@startuml
title Observer: событие «состав расформирован» на сортировочной станции (вариант 7)

interface DismantlingListener {
  +onTrainDismantled(stationName, trainNumber, carCount) : void
  +listenerTitle() : String
}

class Station {
  -stationName : String
  -listeners : List<DismantlingListener>
  +addListener(listener) : void
  +removeListener(listener) : void
  +trainDismantled(trainNumber, carCount) : void
  +deliveryOrder() : List<String>
}

class BoardDisplay
class DismantlingJournal {
  -records : List<String>
  +recordCount() : int
}
class NotificationService
class AsuIntegration

DismantlingListener <|.. BoardDisplay
DismantlingListener <|.. DismantlingJournal
DismantlingListener <|.. NotificationService
DismantlingListener <|.. AsuIntegration

Station o--> DismantlingListener : оповещает
@enduml

In [28]:
!plantuml -tpng observer.puml
from IPython.display import Image
Image("observer.png")

**Вывод по задаче 2 (Observer, вариант 7).** Observer снял связанность: станция больше не знает ни про табло, ни про журнал, ни про рассылку и ни про АСУ — она только уведомляет всех подписанных `DismantlingListener`. Новый наблюдатель добавляется вызовом `addListener` без правки `Station` (проверено на `AsuIntegration`). Порядок получения события определяется порядком подписки: при первом расформировании — `табло → журнал → рассылка`, после отписки табло и подключения АСУ при втором расформировании — `журнал → рассылка → АСУ`.

Цена паттерна: порядок уведомлений стал неявным и зависит от порядка подписки; исключение в одном подписчике прерывает обход остальных (нужен `try/catch` внутри цикла или изолированная доставка); забытая отписка оставляет в субъекте ссылку на объект и мешает сборщику мусора его удалить; уведомления в несколько потоков требуют синхронизации списка слушателей.

### 5.1 Итоговая таблица

| Паттерн | Какую проблему кода «до» устраняет | Что добавляется без правки старого кода | Чем приходится заплатить | Где применим в вашем проекте |
|---|---|---|---|---|
| Strategy | Выбор очерёдности расформирования цепочкой `if` в одном методе `legacyOrder` | Новое правило порядка (например, `BY_PRIORITY`) как новая реализация `DismantlingOrderStrategy`; в образце — рефрижераторная отправка | Дополнительные классы и каталог стратегий; нужно явно выбирать стратегию; расходы на тестирование каждого варианта | Планирование расформирования составов: по назначению, по массе, по типу вагона, по приоритету |
| Factory Method | Разбор строкового типа отцепа в `describe` и создание объектов в вызывающем коде | Новый тип отцепа: `LocalOtsep` и `LocalOtsepCreator` | Рост числа классов; для простого единственного продукта фабрика излишня | Создание отцепов по назначению, транзитных, ремонтных и местных |
| Observer | Станция жёстко знает всех получателей события и печатает их сообщения сама | Новый наблюдатель `AsuIntegration` без изменения `Station` | Неявный порядок доставки, риск ошибки в одном подписчике и утечки памяти при забытой отписке | Оповещение о расформировании состава: табло, журнал, рассылка, АСУ |

### 5.2 Контрольные вопросы

1. **Какой принцип SOLID нарушает разбор типа условным оператором и как паттерны его восстанавливают?**
   **Ответ:** прежде всего **OCP** (принцип открытости/закрытости): новый вариант требует правки работающего метода. Дополнительно нарушается **SRP** — один метод одновременно выбирает вариант и выполняет работу. Strategy выносит каждый алгоритм в отдельный класс, и новый алгоритм добавляется без правки контекста. Factory Method выносит создание конкретного продукта в класс-создатель, и новый продукт добавляется новым создателем. Observer выносит оповещение в интерфейс наблюдателя, и новый получатель добавляется без правки субъекта. Все три восстанавливают OCP; Strategy и Factory Method дополнительно восстанавливают SRP.

2. **Чем Factory Method отличается от статического фабричного метода и от Abstract Factory?**
   **Ответ:** статический фабричный метод — это просто `static`-метод внутри одного класса; он не создаёт иерархию создателей и не может быть полиморфно заменён наследником (нельзя «подменить фабрику»). Factory Method опирается на иерархию `OtsepCreator` и переопределяемый метод `createOtsep`, поэтому клиент работает с абстракцией и конкретный создатель выбирается извне. Abstract Factory возвращает **семейство** связанных продуктов (например, отцеп + наряд + маршрутный лист), а Factory Method — один продукт; Abstract Factory обычно содержит несколько фабричных методов.

3. **Strategy и State имеют почти одинаковую структуру. В чём разница по назначению?**
   **Ответ:** Strategy задаёт **взаимозаменяемый алгоритм**, который клиент выбирает сам и который, как правило, не меняет внутреннее состояние контекста (порядок расформирования — выбор правила сортировки). State описывает **состояние самого объекта**: контекст меняет состояние, и поведение методов зависит от текущего состояния; переходы часто инициируются самим контекстом/состоянием (например, «прибыл», «расформируется», «отправлен»).Структурно оба паттерна делегируют работу через интерфейс, но по смыслу Strategy — это «как делать», а State — «в каком состоянии я нахожусь и что из этого следует».

4. **Почему Observer может привести к утечке памяти и как этого избежать?**
   **Ответ:** субъект хранит сильные ссылки на всех наблюдателей в списке. Если подписчик больше не нужен, но не вызван `removeListener`, ссылка из субъекта не даёт сборщику мусора удалить объект вместе со всей его графой. Избегают так: явная отписка при завершении жизни подписчика (`dispose`), слабые ссылки (`WeakReference`) в списке слушателей, управление жизненным циклом через контейнер/фреймворк и проверка в отладке, что число слушателей не растёт.

5. **Когда введение паттерна вредно? Что такое преждевременное усложнение?**
   **Ответ:** вредно, когда вариантов нет или они никогда не меняются: например, в небольшом скрипте для одного фиксированного порядка расформирования достаточно одного метода; три классаstrategies и каталог только усложняют чтение. То же верно для Observer с единственным получателем и Factory Method с единственным продуктом, создающимся одним `new`. Преждевременное усложнение — это добавление абстракций «на будущее», которого может не наступить: растут число файлов, кривая обучения и стоимость поддержки, а выгоды нет. Паттерн вводят тогда, когда изменение уже реально или предвидится с высокой вероятностью.

### 5.3 Три вывода по работе в целом

1. Разница между кодом «до» и «после» видна не в объёме вычислений, а в том, **где находится точка изменения**: в «до» новый вариант требует правки работающего метода, в «после» — добавления нового класса и регистрации.
2. Все три паттерна заменяют ветвление по типу полиморфизмом, но каждую замену нужно проверять характеризующими тестами: сравнение с эталоном «до» доказывает, что поведение старых сценариев не изменилось.
3. Паттерн — это не «всегда хорошо»: Observer добавляет скрытый порядок доставки и риск утечки, Strategy — каталог и дополнительные классы, Factory Method — иерархию создателей. Их применяют там, где цена усложнения меньше цены будущих правок.

---

## 6. Варианты предметных областей

Номер варианта — по номеру студента в списке группы. Образец (раздел 2) выполняется всеми одинаково; задачи 1 и 2 — на своей предметной области.

| № | Предметная область | Фабрика создаёт | Стратегия варьирует | Наблюдатели получают событие |
|---|---|---|---|---|
| 1 | Грузовая станция | тип вагона | расчёт платы за пользование вагонами | вагон подан под погрузку |
| 2 | Локомотивное депо | вид ремонта (ТО-2, ТР-1, ТР-2) | приоритет заявок на ремонт | локомотив выдан из ремонта |
| 3 | Пассажирская касса | тип проездного документа | начисление скидки | рейс задержан |
| 4 | Контейнерный терминал | тип контейнера | выбор места на площадке | контейнер прибыл |
| 5 | Автотранспортное предприятие | тип транспортного средства | расчёт расхода топлива | путевой лист закрыт |
| 6 | Склад | тип тары | правило размещения (FIFO, ABC) | запас ниже нормы |
| 7 | Сортировочная станция | тип отцепа | очерёдность расформирования | состав расформирован |
| 8 | Пригородные перевозки | тип абонемента | тарификация по зонам | изменено расписание |
| 9 | Пункт таможенного оформления | тип декларации | выбор канала контроля | груз оформлен |
| 10 | Экспедиторская компания | тип заявки (авто, ж. д., море) | выбор перевозчика по критерию | статус отправки изменён |
| 11 | Путевое хозяйство | тип дефекта рельса | назначение вида работ | обнаружен дефект |
| 12 | Энергодиспетчерский пункт | тип потребителя | тарифный план (день, ночь) | авария на подстанции |
| 13 | Вагонное депо | тип неисправности | выбор отцепочного или безотцепочного ремонта | вагон забракован |
| 14 | Логистический центр | тип упаковки | правило консолидации отправок | сформирована отправка |
| 15 | Центр фирменного транспортного обслуживания | тип договора | расчёт скидки за объём | заявка согласована |

## 7. Чек-лист самопроверки

- [x] Раздел 1 выполнен, версия Java не ниже 11 (локальная проверка — JDK 17)
- [x] `TariffBefore.java` и `TariffAfter.java` запущены, вывод сохранён
- [x] Отвечены вопросы к образцу (п. 2.5)
- [x] Дополнительно по варианту 7: `DismantlingBefore.java`, `DismantlingAfter.java`, `strategy_laba7.puml`
- [x] `FactoryAfter.java` написан, все проверки `[OK]`, в `main` нет разбора типов
- [x] Четвёртый тип отцепа (`LocalOtsep`) добавлен новыми классами
- [x] `ObserverAfter.java` написан, все проверки `[OK]`, отписка работает
- [x] Четвёртый наблюдатель (`AsuIntegration`) добавлен без правки класса `Station`
- [x] Построены диаграммы: `strategy.puml`, `strategy_laba7.puml`, `factory.puml`, `observer.puml`
- [x] Заполнена итоговая таблица (п. 5.1)
- [x] Отвечены пять контрольных вопросов и сформулированы три вывода
- [ ] Блокнот выполняется сверху вниз без ошибок после *Среда выполнения → Перезапустить и выполнить все* (проверить в Colab)
- [x] Запущена ячейка самопроверки, её вывод сохранён в `outputs/selfcheck.md` (ФИО и группа указаны)

In [29]:
#@title Запустить самопроверку и получить отчёт
import os, re, subprocess, datetime

ФИО_исполнителя = "Захаренко М.А."  #@param {type:"string"}
Группа_исполнителя = "ГИ-41"  #@param {type:"string"}
Номер_варианта = 7  #@param {type:"integer"}
ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = True  #@param {type:"boolean"}

ПРОГРАММЫ = [
    ("Образец, Strategy", "TariffAfter.java"),
    ("Задача 1, Factory Method", "FactoryAfter.java"),
    ("Задача 2, Observer", "ObserverAfter.java"),
]
ДИАГРАММЫ = ["strategy.puml", "factory.puml", "observer.puml"]

def запустить(имя_файла):
    """Запустить один исходный файл Java и вернуть (успех, пояснение)."""
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    if завершение.returncode != 0:
        значимые = [s for s in вывод.splitlines() if s.strip() and not s.startswith("Picked up")]
        первая_строка = значимые[0] if значимые else "без сообщения"
        return False, "аварийное завершение: " + первая_строка[:90]
    if "[FAIL]" in вывод:
        return False, "проверок не пройдено: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 3:
        return False, "проверок в файле только %d, требуется не менее 3" % успешных
    return True, "проверок пройдено: %d" % успешных

результаты = []
for название, файл in ПРОГРАММЫ:
    результаты.append((название, файл) + запустить(файл))

# Признаки соблюдения требований, которые можно проверить по тексту
замечания = []
def тело_метода(исходный_текст, сигнатура="main("):
    """Вернуть тело метода по сопоставлению фигурных скобок (без построчных комментариев)."""
    без_комментариев = re.sub(r"//[^\n]*", "", исходный_текст)
    начало = без_комментариев.find(сигнатура)
    if начало < 0:
        return ""
    открывающая = без_комментариев.find("{", начало)
    if открывающая < 0:
        return ""
    глубина = 0
    for позиция in range(открывающая, len(без_комментариев)):
        символ = без_комментариев[позиция]
        if символ == "{":
            глубина += 1
        elif символ == "}":
            глубина -= 1
            if глубина == 0:
                return без_комментариев[открывающая:позиция + 1]
    return без_комментариев[открывающая:]

if os.path.exists("FactoryAfter.java"):
    текст_фабрики = open("FactoryAfter.java", encoding="utf-8").read()
    блок_main = тело_метода(текст_фабрики)
    if re.search(r'\b(if|switch)\b', блок_main):
        замечания.append("в FactoryAfter.java метод main содержит ветвление по типу")
    if текст_фабрики.count("class ") < 5:
        замечания.append("в FactoryAfter.java меньше пяти классов: продукты, создатели и четвёртый тип, вероятно, не описаны")
if os.path.exists("ObserverAfter.java"):
    текст_наблюдателя = open("ObserverAfter.java", encoding="utf-8").read()
    for требуемое in ("addListener", "removeListener"):
        if требуемое not in текст_наблюдателя:
            замечания.append("в ObserverAfter.java нет метода %s" % требуемое)
отсутствующие_диаграммы = [d for d in ДИАГРАММЫ if not os.path.exists(d)]

строки = [
    "### Отчёт самопроверки «Песочница паттернов GoF»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s, вариант %d |" % (
        ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА", Номер_варианта),
    "| Дата и время отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "| Версия Java | %s |" % next(
        (s for s in subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()
         if "version" in s), "не определена"),
    "",
    "| Программа | Файл | Результат |",
    "|---|---|---|",
]
for название, файл, успех, пояснение in результаты:
    строки.append("| %s | `%s` | %s — %s |" % (название, файл, "пройдено" if успех else "НЕ ПРОЙДЕНО", пояснение))
строки += [
    "",
    "| Диаграммы | %s |" % ("все три на месте" if not отсутствующие_диаграммы
                            else "отсутствуют: " + ", ".join(отсутствующие_диаграммы)),
    "| Таблица и выводы | %s |" % ("заполнены" if ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
    "| Замечания | %s |" % ("нет" if not замечания else "; ".join(замечания)),
]
print("\n".join(строки))
print("\nСкопируйте отчёт выше в описание pull request.\n")

ошибки = []
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
ошибки += ["%s: %s" % (название, пояснение) for название, файл, успех, пояснение in результаты if not успех]
if отсутствующие_диаграммы:
    ошибки.append("нет диаграмм: " + ", ".join(отсутствующие_диаграммы))
if not ТАБЛИЦА_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены итоговая таблица и выводы")
ошибки += замечания

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")

### Отчёт самопроверки «Песочница паттернов GoF»

| Параметр | Значение |
|---|---|
| Исполнитель | Захаренко М.А., группа ГИ-41, вариант 7 |
| Дата и время отчёта (UTC) | 2026-11-20 09:11 |
| Версия Java | openjdk version "17.0.21" 2026-10-20 |

| Программа | Файл | Результат |
|---|---|---|
| Образец, Strategy | `TariffAfter.java` | пройдено — проверок пройдено: 5 |
| Задача 1, Factory Method | `FactoryAfter.java` | пройдено — проверок пройдено: 9 |
| Задача 2, Observer | `ObserverAfter.java` | пройдено — проверок пройдено: 7 |

| Диаграммы | все три на месте |
| Таблица и выводы | заполнены |
| Замечания | нет |

Скопируйте отчёт выше в описание pull request.

Самопроверка пройдена: работа готова к сдаче.


---

## 9. Критерии оценивания

| Критерий | Баллов |
|---|---|
| Образец запущен, вопросы п. 2.5 отвечены содержательно | 10 |
| Factory Method: корректная структура паттерна, в `main` нет разбора типов | 20 |
| Factory Method: четвёртый тип добавлен новым классом | 10 |
| Observer: корректная структура, работают подписка и отписка | 20 |
| Observer: четвёртый наблюдатель добавлен без правки субъекта | 10 |
| Проверки в каждом файле осмысленны (не `[OK]` ради `[OK]`) | 10 |
| Три диаграммы классов соответствуют коду | 10 |
| Итоговая таблица, контрольные вопросы, выводы | 10 |
| **Итого** | **100** |

Снижение: задача решена не на своём варианте — минус 20; код «после» повторяет разбор типа условным оператором внутри нового класса — минус 20; диаграмма не соответствует коду — минус 10.

## 10. Для желающих (сверх задания)

| Что сделать | Зачем |
|---|---|
| Подключить настоящий JUnit 5 вместо метода `check`: `!wget -q https://repo1.maven.org/maven2/org/junit/platform/junit-platform-console-standalone/1.10.2/junit-platform-console-standalone-1.10.2.jar` и запуск через `java -jar` | переход к инструменту, который используется в ЛР по модульному тестированию |
| Заменить три `if` в выборе стратегии на `java.util.Map` с ключом-перечислением (`enum`) | показать, что каталог стратегий сам может быть устранён из вызывающего кода |
| Реализовать Observer на `java.util.function.Consumer` вместо своего интерфейса | сравнить классический GoF и функциональный стиль |